# ISOM5240 Fine-tuning Notebook — Pipeline 2: Freshness Detection

**Workflow:**
- Phase 1 → Quick 1-epoch fine-tune of 3 models on small subset → fair comparison → select best
- Phase 2 → Full fine-tune of selected model on complete dataset
- Phase 3 → Evaluate (accuracy, precision, recall, confusion matrix)
- Phase 4 → Export Excel + push to HuggingFace Hub

**Dataset:** Kaggle "Fruits Fresh and Rotten for Classification"  
**Models:** ViT-base / ResNet-50 / Swin-tiny  
**Task:** Binary image classification (fresh vs rotten)

## Step 1: Install dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate pillow scikit-learn -q
!pip install huggingface_hub -q

## Step 2: GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → T4 GPU → Save, then Run All again."
    )

device = torch.device("cuda")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Login to HuggingFace Hub

Add `HF_TOKEN` in Colab Secrets (🔑 left sidebar) before running.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in to HuggingFace Hub")

## Step 4: Download dataset from Kaggle

Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets before running.

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification
!unzip -q fruits-fresh-and-rotten-for-classification.zip

# Auto-detect the folder containing train/ and test/
import glob
for d in glob.glob("**/train", recursive=True):
    DATA_DIR = os.path.dirname(d)
    break

print(f"DATA_DIR = {DATA_DIR}")
print(f"Train folders: {os.listdir(os.path.join(DATA_DIR, 'train'))}")
print(f"Test folders: {os.listdir(os.path.join(DATA_DIR, 'test'))}")

In [ ]:
# Remove duplicate outer train/test to avoid double-counting
!rm -rf dataset/train dataset/test

## Step 5: Load dataset and re-label to binary

Original 6 classes → 2 classes: fresh (0) / rotten (1)

In [ ]:
from datasets import load_dataset, DatasetDict
import numpy as np
import time

dataset = load_dataset("imagefolder", data_dir=DATA_DIR, drop_labels=False)

print("Loaded dataset:")
print(dataset)
print(f"Original labels: {dataset['train'].features['label'].names}")

In [ ]:
# Re-label: 6 classes → 2 classes
original_names = dataset["train"].features["label"].names

label_mapping = {}
for idx, name in enumerate(original_names):
    label_mapping[idx] = 0 if "fresh" in name.lower() else 1

print(f"Mapping: {label_mapping}")

def relabel(example):
    example["label"] = label_mapping[example["label"]]
    return example

dataset = dataset.map(relabel)
LABEL_NAMES = ["fresh", "rotten"]

train_labels = np.array(dataset["train"]["label"])
test_labels = np.array(dataset["test"]["label"])
print(f"Train: {len(train_labels)} | fresh: {(train_labels==0).sum()} | rotten: {(train_labels==1).sum()}")
print(f"Test:  {len(test_labels)} | fresh: {(test_labels==0).sum()} | rotten: {(test_labels==1).sum()}")

## Step 6: Split validation set (10% from train)

In [ ]:
split = dataset["train"].train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": dataset["test"],
})

print(f"Train:      {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test:       {len(dataset['test'])}")

---
# PHASE 1: Model Selection

Quick 1-epoch fine-tune of each model on a **small subset (500 samples)** to fairly compare accuracy and runtime on the same task. This is necessary because the pre-trained models output ImageNet labels (1000 classes), not fresh/rotten, so we cannot compare them without fine-tuning.

## Step 7: Define candidate models

In [ ]:
CANDIDATE_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

print("Candidates:")
for name, path in CANDIDATE_MODELS.items():
    print(f"  {name}: {path}")

## Step 8: Quick 1-epoch comparison on small subset

In [ ]:
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    pipeline as hf_pipeline,
)
import evaluate
import pandas as pd
import glob
from PIL import Image as PILImage

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Small subset for quick comparison
small_train = dataset["train"].select(range(500))
small_test = dataset["test"].select(range(200))

quick_results = []

for model_key, model_path in CANDIDATE_MODELS.items():
    print(f"\n{'='*50}")
    print(f"Quick training: {model_key}")
    print(f"{'='*50}")

    # Load processor and set transform
    proc = AutoImageProcessor.from_pretrained(model_path)

    def make_preprocess(processor):
        def preprocess(batch):
            images = [img.convert("RGB") for img in batch["image"]]
            inputs = processor(images=images, return_tensors="pt")
            inputs["label"] = batch["label"]
            return inputs
        return preprocess

    transform_fn = make_preprocess(proc)
    small_train.set_transform(transform_fn)
    small_test.set_transform(transform_fn)

    # Load model with 2-class head
    mdl = AutoModelForImageClassification.from_pretrained(
        model_path,
        num_labels=2,
        id2label={0: "fresh", 1: "rotten"},
        label2id={"fresh": 0, "rotten": 1},
        ignore_mismatched_sizes=True,
    )

    total_params = sum(p.numel() for p in mdl.parameters())

    args = TrainingArguments(
        output_dir=f"./quick-{model_key}",
        num_train_epochs=1,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        fp16=True,
        remove_unused_columns=False,
        logging_steps=999,
        report_to="none",
    )

    trainer = Trainer(
        model=mdl, args=args,
        train_dataset=small_train,
        eval_dataset=small_test,
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    results = trainer.evaluate()

    # Measure inference speed
    test_pipe = hf_pipeline("image-classification", model=mdl, image_processor=proc, device=0)
    inf_paths = (glob.glob(f"{DATA_DIR}/test/**/*.jpg", recursive=True) +
                 glob.glob(f"{DATA_DIR}/test/**/*.png", recursive=True) +
                 glob.glob(f"{DATA_DIR}/test/**/*.jpeg", recursive=True))[:50]
    inf_times = []
    for path in inf_paths:
        img = PILImage.open(path).convert("RGB")
        t0 = time.time()
        test_pipe(img)
        inf_times.append(time.time() - t0)
    avg_inf_ms = np.mean(inf_times) * 1000

    quick_results.append({
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "1-Epoch Accuracy": round(results["eval_accuracy"], 4),
        "Train Time (s)": round(train_time, 1),
        "Avg Inference (ms)": round(avg_inf_ms, 1),
    })

    print(f"  Accuracy: {results['eval_accuracy']:.4f} | Time: {train_time:.0f}s | Speed: {avg_inf_ms:.1f}ms | Params: {total_params/1e6:.1f}M")

df_selection = pd.DataFrame(quick_results)
print("\n" + "="*60)
print("PHASE 1 RESULTS: Model Selection (1-epoch, 500 samples)")
print("="*60)
print(df_selection.to_string(index=False))

In [ ]:
import glob
all_files = glob.glob(f"{DATA_DIR}/test/**/*.*", recursive=True)
print(f"Total files: {len(all_files)}")
if all_files:
    # Show unique extensions
    exts = set(os.path.splitext(f)[1].lower() for f in all_files)
    print(f"Extensions: {exts}")
    print(f"Sample: {all_files[:3]}")

## Step 9: Select the best model

**Update `SELECTED_MODEL_KEY` based on Phase 1 results above.**

In [ ]:
# Auto-select the model with highest accuracy
best = max(quick_results, key=lambda x: x["1-Epoch Accuracy"])
SELECTED_MODEL_KEY = best["Model"]
SELECTED_MODEL_PATH = CANDIDATE_MODELS[SELECTED_MODEL_KEY]

print(f"Selected: {SELECTED_MODEL_KEY} (Accuracy: {best['1-Epoch Accuracy']})")
print(f"Path: {SELECTED_MODEL_PATH}")

---
# PHASE 2: Full Fine-tuning of Selected Model

## Step 10: Prepare data for selected model

In [ ]:
processor = AutoImageProcessor.from_pretrained(SELECTED_MODEL_PATH)

def preprocess(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["label"] = batch["label"]
    return inputs

dataset["train"].set_transform(preprocess)
dataset["validation"].set_transform(preprocess)
dataset["test"].set_transform(preprocess)

print(f"Transform set for {SELECTED_MODEL_KEY}")

## Step 11: Load model with 2-class head

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    SELECTED_MODEL_PATH,
    num_labels=len(LABEL_NAMES),
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {SELECTED_MODEL_KEY}")
print(f"Total: {total_params:,} | Trainable: {trainable:,}")

## Step 12: Training configuration

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./freshness-{SELECTED_MODEL_KEY.lower()}",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    remove_unused_columns=False,
    fp16=True,
    dataloader_num_workers=2,
    push_to_hub=False,
)

print("Config: 3 epochs, lr=2e-5, batch=32, fp16=True")

## Step 13: Train!

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start

print(f"\nTraining complete in {train_time/60:.1f} minutes")

## Step 14: Evaluate on test set — Accuracy

In [ ]:
ft_results = trainer.evaluate(dataset["test"])
print(f"Test Accuracy: {ft_results['eval_accuracy']:.4f}")
print(f"Test Loss:     {ft_results['eval_loss']:.4f}")

## Step 15: Evaluate on test set — Precision, Recall, F1, Confusion Matrix

In [ ]:
from transformers import pipeline as hf_pipeline
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image as PILImage

pipe_ft = hf_pipeline("image-classification", model=model, image_processor=processor, device=0)

test_paths = glob.glob(f"{DATA_DIR}/test/**/*.jpg", recursive=True) + \
             glob.glob(f"{DATA_DIR}/test/**/*.png", recursive=True) + \
             glob.glob(f"{DATA_DIR}/test/**/*.jpeg", recursive=True)

y_true = []
y_pred = []

for path in test_paths:
    img = PILImage.open(path).convert("RGB")
    folder = os.path.basename(os.path.dirname(path)).lower()
    true_label = "fresh" if "fresh" in folder else "rotten"

    pred = pipe_ft(img, top_k=1)
    pred_label = pred[0]["label"]

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=["fresh", "rotten"])
print(cm)
print(f"\n[[TN={cm[0,0]}, FP={cm[0,1]}]")
print(f" [FN={cm[1,0]}, TP={cm[1,1]}]]")

## Step 16: Inference speed measurement

In [ ]:
ft_times = []
for path in test_paths[:100]:
    img = PILImage.open(path).convert("RGB")
    t0 = time.time()
    pipe_ft(img)
    ft_times.append(time.time() - t0)

ft_avg_ms = np.mean(ft_times) * 1000
print(f"Avg inference: {ft_avg_ms:.1f}ms per image")

## Step 17: Summary table — Before vs After

In [ ]:
# Find Phase 1 result for selected model
selected_phase1 = next(r for r in quick_results if r["Model"] == SELECTED_MODEL_KEY)

comparison = pd.DataFrame([
    {
        "Stage": "Pre-trained (1-epoch, 500 samples)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": selected_phase1["1-Epoch Accuracy"],
        "Loss": "N/A",
        "Avg Inference (ms)": "N/A",
    },
    {
        "Stage": "Fine-tuned (full dataset)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": round(ft_results["eval_accuracy"], 4),
        "Loss": round(ft_results["eval_loss"], 4),
        "Avg Inference (ms)": round(ft_avg_ms, 1),
    },
])

print("="*60)
print("Before vs After Fine-tuning")
print("="*60)
print(comparison.to_string(index=False))

## Step 18: Export experiment results to Excel

In [ ]:
with pd.ExcelWriter("Experimental_results.xlsx") as writer:
    df_selection.to_excel(writer, sheet_name="P2 Model Selection", index=False)
    comparison.to_excel(writer, sheet_name="P2 Fine-tune Result", index=False)

print("Saved to Experimental_results.xlsx")

from google.colab import files
files.download("Experimental_results.xlsx")
print("Downloaded!")

## Step 19: Push fine-tuned model to HuggingFace Hub

In [ ]:
HUB_MODEL_ID = "Alisa-Sun/freshness-detection"

# Push model directly (not via trainer)
model.push_to_hub(HUB_MODEL_ID, commit_message=f"Fine-tuned {SELECTED_MODEL_KEY} | acc={ft_results['eval_accuracy']:.4f}")
processor.push_to_hub(HUB_MODEL_ID)

print(f"Model pushed to: https://huggingface.co/{HUB_MODEL_ID}")

## Step 20: Final inference test

In [ ]:
pipe_final = hf_pipeline("image-classification", model=HUB_MODEL_ID)

test_paths = (glob.glob(f"{DATA_DIR}/test/**/*.jpg", recursive=True) +
              glob.glob(f"{DATA_DIR}/test/**/*.png", recursive=True) +
              glob.glob(f"{DATA_DIR}/test/**/*.jpeg", recursive=True))

print(f"Testing 5 images from test set ({len(test_paths)} total):")
for path in test_paths[:5]:
    img = PILImage.open(path).convert("RGB")
    folder = os.path.basename(os.path.dirname(path)).lower()
    true_label = "fresh" if "fresh" in folder else "rotten"

    pred = pipe_final(img)
    pred_label = pred[0]["label"]
    pred_score = pred[0]["score"]
    status = "✅" if pred_label == true_label else "❌"
    print(f"  {status} True: {true_label} | Pred: {pred_label} ({pred_score:.3f})")

---
## Notebook Summary

| Step | Content |
|------|---------|
| 1-3 | Setup: dependencies, GPU check, HuggingFace login |
| 4-6 | Data: download Kaggle dataset, re-label, split validation |
| 7-9 | **Phase 1:** Quick 1-epoch fine-tune of ViT/ResNet/Swin on 500 samples → select best |
| 10-13 | **Phase 2:** Full fine-tune of selected model on complete dataset (3 epochs) |
| 14-16 | Evaluate: accuracy, precision, recall, F1, confusion matrix, inference speed |
| 17 | Summary table: before vs after fine-tuning |
| 18 | Export Excel + auto-download |
| 19-20 | Push to HuggingFace Hub + final inference test |